![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**Module 08: Advanced Agentic AI**

---

## Session 8E: Agent Hooks and Workflow Guards

<div align="center">

<table>
<thead><tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr></thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Mandatory part</td><td>Local hook engine for pre-tool, post-tool and stop events</td></tr>
<tr><td align="left">Main output</td><td>A controlled hook system that blocks unsafe calls and audits tool use</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m08e-overview)
2. [Setup and Background](#m08e-setup)
3. [Core Concepts](#m08e-concepts)
4. [Guided Implementation](#m08e-implementation)
5. [Testing and Analysis](#m08e-testing)
6. [Student Tasks](#m08e-tasks)
7. [Submission and Reflection](#m08e-submission)

---

<a id="m08e-overview"></a>

### 1. Overview and Learning Goals

A hook is a function or command that runs automatically at a specific workflow event. In agentic AI, hooks guard tool use, inspect outputs, log actions and run final checks. Think of them as security checkpoints in an airport: the traveller (a tool call) passes a screening gate before boarding (PreToolUse), a customs check after landing (PostToolUse), and a final exit control before leaving the building (Stop). The traveller does not choose whether to be screened, and that is exactly the point — hooks run whether or not the agent "remembers" to be careful.

The full lifecycle you will build looks like this:

```text
User request
     |
     v
Agent decides to call a tool
     |
     v
PreToolUse hook ------ blocked ------> refusal recorded in audit log,
     |                                  tool NEVER runs
     | allowed
     v
Tool executes
     |
     v
PostToolUse hook  (log result metadata, check output format)
     |
     v
Final response
     |
     v
Stop hook  (final audit summary before the workflow finishes)
```

Hooks are not a replacement for careful tool design. They are an additional control layer: even a well-designed tool benefits from an independent gate that can refuse, log and summarise. By the end of this session you should be able to explain each hook event, implement a guard that returns an explicit allow/block decision, wire guards into a tool runner, and read an audit log to reconstruct exactly what the system did and why.

<a id="m08e-setup"></a>

### 2. Setup and Background

The whole session runs on the Python standard library, so it works identically in Google Colab and local Jupyter with nothing to install and no API keys. This is deliberate: hook systems in real agent frameworks (for example coding-agent hooks) are configuration around an opaque core, whereas here you build the core yourself, so nothing about the control flow is hidden from you.

The safety boundary is the standard Module 08 boundary: synthetic teaching data only, no private documents or credentials, no real external side effects, and every decision must leave an inspectable trace. Notice that this notebook does not merely *follow* the boundary — it *implements* one. The blocked-tool and blocked-term lists you will meet in Section 4 are a working, machine-enforced version of the same rules.

Run the setup cell below first. If any later cell raises `NameError`, restart the kernel and run all cells in order — the classes build on one another.

In [ ]:
# Standard-library imports only: the hook engine is fully offline by design.
import json                                   # serialising arguments for term screening
from dataclasses import dataclass, field     # small, typed records for decisions and audits
from typing import Any, Callable, Dict, List # hook signatures are documented via type hints

print("M08E hooks setup complete.")

<a id="m08e-concepts"></a>

### 3. Core Concepts

Every hook, regardless of framework, follows one pattern: an event fires, a guard examines the event payload, and the guard returns a decision that either lets the workflow continue or stops it with a reason. Everything else is detail.

```text
event occurs --> guard(payload) --> decision
                                      |
                                      |-- allowed: workflow continues,
                                      |            audit record written
                                      |
                                      '-- blocked: workflow stops here,
                                                   reason + audit record written
```

Common hook events:

<div align="center">

<table>
<thead><tr><th><strong>Hook event</strong></th><th><strong>When it runs</strong></th><th><strong>Example use</strong></th></tr></thead>
<tbody>
<tr><td align="left">PreToolUse</td><td>Before a tool runs.</td><td>Block shell commands or private-file access.</td></tr>
<tr><td align="left">PostToolUse</td><td>After a tool runs.</td><td>Log result metadata or check output format.</td></tr>
<tr><td align="left">Stop</td><td>Before the workflow finishes.</td><td>Run final safety or audit checks.</td></tr>
</tbody>
</table>

</div>

A hook should be simple and predictable. It should check a concrete rule and return a concrete decision. Resist the temptation to make guards clever: a guard that is hard to reason about is itself a risk, because you can no longer say with confidence what it will block. Three design rules follow from this: a guard must always return a decision (never raise on ordinary input), a blocked decision must carry a human-readable reason, and every decision — allowed or blocked — must be recorded so the audit log tells the complete story, not just the exceptions.

<a id="m08e-implementation"></a>

### 4. Guided Implementation

You will build the system bottom-up in four steps: the record types, the engine that dispatches events, the guard functions, and a tool runner that puts the gates in front of real (safe, deterministic) tools. Read each explanation, predict the behaviour, then run the cell.

#### 4.1 Decision and Audit Records

Three small dataclasses define the vocabulary of the whole system. `HookDecision` is what every guard must return: an explicit boolean, a human-readable reason, and optional metadata. `ToolCall` describes what the agent wants to do. `AuditRecord` is one line of history: which event fired, for which tool, what was decided and why. Using dataclasses rather than loose dictionaries is a design decision — a guard that forgets to supply a reason fails immediately at construction time instead of producing a silent, unexplained log entry later.

In [ ]:
@dataclass
class HookDecision:
    # Every guard MUST return one of these: an explicit verdict plus a
    # human-readable reason. No implicit None-means-allowed shortcuts.
    allowed: bool
    reason: str
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class ToolCall:
    # What the agent wants to do: a tool name and its arguments.
    tool_name: str
    args: Dict[str, Any]

@dataclass
class AuditRecord:
    # One line of history. Records are written for allowed AND blocked
    # decisions, so the audit log tells the whole story, not just failures.
    event: str
    tool_name: str
    allowed: bool
    reason: str
    metadata: Dict[str, Any] = field(default_factory=dict)

#### 4.2 The Hook Engine

The engine is a small event dispatcher. It keeps a registry of guard functions per event, and `run_event` calls each registered guard in order. Two behaviours deserve attention. First, *fail-fast*: the first guard that blocks ends the event immediately — later guards never run, exactly like airport security turning a traveller away at the first failed checkpoint. Second, *audit-everything*: each guard's decision is appended to the audit log before the engine decides what to do next, so even a blocked call leaves a complete trace. Registering a guard for an unknown event raises immediately; asking to run an unknown event returns a blocked decision rather than crashing, because at run time the safe answer to "I do not understand this situation" is "do not proceed".

In [ ]:
class HookEngine:
    def __init__(self):
        # One guard list per supported event. The audit log is engine-wide
        # so a single read shows the full history across all events.
        self.hooks = {"PreToolUse": [], "PostToolUse": [], "Stop": []}
        self.audit_log: List[AuditRecord] = []

    def register(self, event: str, hook_fn: Callable[[Dict[str, Any]], HookDecision]) -> None:
        # Registration-time errors should be loud: a typo in an event name
        # must fail here, not silently create a guard that never runs.
        if event not in self.hooks:
            raise ValueError(f"Unknown hook event: {event}")
        self.hooks[event].append(hook_fn)

    def run_event(self, event: str, payload: Dict[str, Any]) -> HookDecision:
        # Run-time errors should be safe: an unknown event blocks rather
        # than crashes, because "not understood" must never mean "allowed".
        if event not in self.hooks:
            return HookDecision(False, f"Unknown hook event: {event}")

        for hook_fn in self.hooks[event]:
            decision = hook_fn(payload)
            tool_name = payload.get("tool_name", "none")
            # Audit BEFORE acting on the decision: even blocked calls leave
            # a complete trace of what was checked and why.
            self.audit_log.append(AuditRecord(event, tool_name, decision.allowed, decision.reason, decision.metadata))
            if not decision.allowed:
                return decision   # fail fast: later guards never run

        return HookDecision(True, f"All {event} hooks passed.")

#### 4.3 Guard Functions

Three guards cover the three events. The PreToolUse guard is the real gatekeeper and applies two independent screens: a denylist of tool *names* that must never run (shell access, email, credential reads), and a scan of the *arguments* for sensitive terms, so a harmless-looking tool cannot be used to smuggle a password through. Serialising the arguments to JSON before scanning is a small but deliberate trick — it flattens nested structures so a term hidden two levels deep is still caught. The PostToolUse and Stop guards always allow in this lab; their job is to attach useful metadata to the audit trail. That is realistic: most hooks observe, few block, but the blocking ones must be reliable.

Expected behaviour to verify later: a call to `rectangle_area` passes both screens; a call to `shell` is blocked by name; a `word_count` call whose text contains "password" is blocked by the argument screen even though the tool itself is safe.

In [ ]:
def pre_tool_safety_hook(payload: Dict[str, Any]) -> HookDecision:
    tool_name = payload.get("tool_name", "")
    args = payload.get("args", {})
    # Screen 1: some tools are refused by NAME, whatever their arguments.
    blocked_tools = {"shell", "send_email", "read_private_file", "access_credentials"}

    if tool_name in blocked_tools:
        return HookDecision(False, f"Blocked unsafe tool: {tool_name}", {"blocked_tool": tool_name})

    # Screen 2: safe tools can still carry unsafe ARGUMENTS. JSON-encoding
    # flattens nested args so sensitive terms cannot hide inside structures.
    arg_text = json.dumps(args).lower()
    blocked_terms = ["password", "api_key", "credential", "private file", "student record", "hidden solution"]

    if any(term in arg_text for term in blocked_terms):
        return HookDecision(False, "Blocked arguments containing sensitive or private terms.", {"args": args})

    return HookDecision(True, "Pre-tool safety check passed.")

def post_tool_audit_hook(payload: Dict[str, Any]) -> HookDecision:
    # Observe-only guard: always allows, but enriches the audit trail with
    # the result type so anomalies (e.g. unexpected None) are visible later.
    return HookDecision(True, "Post-tool audit recorded.", {"result_type": type(payload.get("result")).__name__})

def stop_review_hook(payload: Dict[str, Any]) -> HookDecision:
    # Final checkpoint: summarise how many calls were blocked in this run.
    return HookDecision(True, "Workflow stopped with audit summary.", {"blocked_calls": payload.get("blocked_calls", 0)})

#### 4.4 A Hooked Tool Runner

The runner is where the gates meet the tools. `run_tool` follows the lifecycle from Section 1 exactly: PreToolUse first, and if that blocks, the tool is never executed and the refusal is returned as a *successful* structured result — being blocked is the system working, not an error. Only after the gate passes does the runner check that the tool exists, execute it inside a `try/except` so a crashing tool cannot take down the workflow, and then fire PostToolUse. The `stop` method fires the Stop hook with the count of blocked calls, giving the run a final, auditable summary.

Note the status vocabulary in the results: `blocked` (refused by a guard), `completed` (ran and passed post-checks), and a plain `ok: False` error for unknown tools or execution failures. Keeping these outcomes distinct is what lets tests — and humans — tell "refused by policy" apart from "broken".

The demonstration cell then wires everything together: register the three guards, run one allowed call, one blocked call and one more allowed call, and finish with `stop`. Predict the four outputs before running it.

In [ ]:
# The tools themselves are tiny and deterministic on purpose: all the
# interesting behaviour in this lab lives in the guards around them.

def rectangle_area(width: float, height: float) -> float:
    return width * height

def safe_word_count(text: str) -> int:
    return len(text.split())

def uppercase_tool(args):
    return str(args["text"]).upper()

# Explicit registry: only tools listed here can run at all. The lambdas
# coerce argument types at the boundary so tools receive clean inputs.
TOOLS = {
    "rectangle_area": lambda args: rectangle_area(float(args["width"]), float(args["height"])),
    "word_count": lambda args: safe_word_count(str(args["text"])),
    "uppercase": uppercase_tool,
}

class HookedToolRunner:
    def __init__(self, hook_engine: HookEngine, tools: Dict[str, Callable[[Dict[str, Any]], Any]]):
        self.hook_engine = hook_engine
        self.tools = tools
        self.blocked_calls = 0   # running tally, reported by the Stop hook

    def run_tool(self, tool_call: ToolCall) -> Dict[str, Any]:
        # Gate first, always. If PreToolUse blocks, the tool NEVER executes,
        # and the refusal is a structured result, not an exception: being
        # blocked is the system working as designed.
        pre = self.hook_engine.run_event("PreToolUse", {"tool_name": tool_call.tool_name, "args": tool_call.args})
        if not pre.allowed:
            self.blocked_calls += 1
            return {"ok": True, "result": {"status": "blocked", "reason": pre.reason}}

        # Unknown tools are an error (caller mistake), distinct from a block.
        if tool_call.tool_name not in self.tools:
            self.blocked_calls += 1
            return {"ok": False, "error": f"Unknown tool: {tool_call.tool_name}", "result": None}

        # Execute defensively: a crashing tool must not take down the runner.
        try:
            result = self.tools[tool_call.tool_name](tool_call.args)
        except Exception as exc:
            return {"ok": False, "error": f"Tool execution failed: {exc}", "result": None}

        # Post-check fires only for tools that actually ran.
        post = self.hook_engine.run_event("PostToolUse", {"tool_name": tool_call.tool_name, "args": tool_call.args, "result": result})
        return {"ok": True, "result": {"status": "completed", "tool_result": result, "post_hook": post.reason}}

    def stop(self) -> Dict[str, Any]:
        # Final checkpoint: hand the blocked-call tally to the Stop hook so
        # every run ends with an auditable summary.
        decision = self.hook_engine.run_event("Stop", {"blocked_calls": self.blocked_calls})
        return {"ok": decision.allowed, "result": {"summary": decision.reason, "blocked_calls": self.blocked_calls}}

In [ ]:
# Wire everything together and walk the full lifecycle once.
engine = HookEngine()
engine.register("PreToolUse", pre_tool_safety_hook)
engine.register("PostToolUse", post_tool_audit_hook)
engine.register("Stop", stop_review_hook)

runner = HookedToolRunner(engine, TOOLS)
print(runner.run_tool(ToolCall("rectangle_area", {"width": 3, "height": 4})))   # expected: completed, 12.0
print(runner.run_tool(ToolCall("shell", {"command": "rm -rf /"})))               # expected: blocked by name
print(runner.run_tool(ToolCall("uppercase", {"text": "safe hook"})))             # expected: completed, "SAFE HOOK"
print(runner.stop())                                                             # expected: summary, blocked_calls=1

#### 4.5 Inspection and Audit Logs

A hook system is only as trustworthy as its audit trail. The log should show what was checked, what was allowed, what was blocked and why — for every event, not only the dramatic ones. When you read the output below, verify three things: every tool call has a PreToolUse line; only calls that actually executed have a PostToolUse line (the blocked `shell` call must not have one); and each blocked line carries a reason specific enough that someone who was not present could reconstruct the incident. If any of those checks fail, the guard or the engine has a gap.

In [ ]:
def display_audit_log(engine: HookEngine) -> None:
    # One line per decision, oldest first: a readable incident timeline.
    for record in engine.audit_log:
        print(f"{record.event} | tool={record.tool_name} | allowed={record.allowed} | reason={record.reason}")
        if record.metadata:
            print("  metadata:", record.metadata)

display_audit_log(engine)

#### 4.6 Optional Coding-Agent Hook Mapping

The local hook engine maps conceptually onto the hook systems of real coding agents, where hooks are shell commands or scripts configured to run at the same lifecycle points. Nothing needs to be installed for this lab; the value of the mapping is recognising that what you just built by hand is what those systems provide as configuration:

<div align="center">

<table>
<thead><tr><th><strong>Local hook</strong></th><th><strong>Coding-agent use</strong></th></tr></thead>
<tbody>
<tr><td align="left">PreToolUse</td><td>Block unsafe shell commands or protected-file edits.</td></tr>
<tr><td align="left">PostToolUse</td><td>Log modified files or run formatting checks.</td></tr>
<tr><td align="left">Stop</td><td>Run final scan or remind user to review changes.</td></tr>
</tbody>
</table>

</div>

The design rules carry over unchanged: guards must be simple and predictable, blocked decisions need reasons, and everything must be audited. If you later configure hooks in a real coding agent, start from the same denylist-plus-argument-screen pattern you implemented here.

In [ ]:
# No real hook-system configuration is required for this lab; the mapping
# above is conceptual. This placeholder keeps the section runnable.
print("Optional real hook-system configuration is not required for this lab.")

<a id="m08e-testing"></a>

### 5. Testing and Analysis

The tests build a fresh engine and runner so results never depend on the demonstration cells above — test isolation is itself a lesson: shared state makes green tests lie. The asserts then pin down every behaviour class from the lifecycle: an allowed call completes with the right value (normal); a denylisted tool and a sensitive-argument call are both blocked with `status: "blocked"` (safety); an unknown tool is an error, not a block (failure); the Stop hook reports at least two blocked calls; and the audit log contains records for all of it. Read each failing assert, if any, as naming the exact gate that broke: a failing normal case means the pipeline is damaged, while a failing blocked case means the safety layer has a hole — the second is worse precisely because nothing visibly crashes when it opens.

In [ ]:
# Fresh engine and runner: tests must not inherit state (audit records,
# blocked-call counts) from the demonstration cells above.
test_engine = HookEngine()
test_engine.register("PreToolUse", pre_tool_safety_hook)
test_engine.register("PostToolUse", post_tool_audit_hook)
test_engine.register("Stop", stop_review_hook)
test_runner = HookedToolRunner(test_engine, TOOLS)

# Normal case: an allowed call runs and returns the correct value.
allowed = test_runner.run_tool(ToolCall("rectangle_area", {"width": 6, "height": 7}))
assert allowed["ok"] is True
assert allowed["result"]["status"] == "completed"
assert allowed["result"]["tool_result"] == 42

# Safety case 1: a denylisted tool NAME is blocked before execution.
blocked_tool = test_runner.run_tool(ToolCall("send_email", {"to": "x@example.com"}))
assert blocked_tool["ok"] is True                      # blocking is success, not error
assert blocked_tool["result"]["status"] == "blocked"

# Safety case 2: a safe tool with sensitive ARGUMENTS is also blocked.
blocked_args = test_runner.run_tool(ToolCall("word_count", {"text": "my password is 123"}))
assert blocked_args["ok"] is True
assert blocked_args["result"]["status"] == "blocked"

# Normal case: another allowed tool passes both screens.
uppercase = test_runner.run_tool(ToolCall("uppercase", {"text": "safe"}))
assert uppercase["ok"] is True
assert uppercase["result"]["tool_result"] == "SAFE"

# Failure case: an unknown tool is a caller error (ok False), distinct
# from a policy block (ok True + status blocked).
unknown = test_runner.run_tool(ToolCall("unknown_tool", {}))
assert unknown["ok"] is False

# Stop hook: the final summary must report the blocked calls above.
stop = test_runner.stop()
assert stop["ok"] is True
assert stop["result"]["blocked_calls"] >= 2

# Audit trail: every decision, allowed or blocked, left a record.
assert len(test_engine.audit_log) >= 4

print("All M08E mandatory hook tests passed.")

<a id="m08e-tasks"></a>

### 6. Student Tasks

Complete the tasks below using only the local hook engine. For each programming task, state the expected normal, edge and failure behaviour before you write code, then prove it with tests.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What you need to do</strong></th><th><strong>Why it matters</strong></th><th><strong>Expected evidence</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline tests</td><td>Run all mandatory cells from the top. Normal: every assert passes. Failure: a <code>NameError</code> means cells ran out of order — restart and run all.</td><td>Confirms the reference behaviour before you change anything.</td><td>Output showing <code>All M08E mandatory hook tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add a new PreToolUse rule</td><td>Extend <code>pre_tool_safety_hook</code> (or register a second guard) to block one additional unsafe argument pattern, for example a URL, a file path outside the workspace, or an email address. Edge: decide how near-misses (e.g. the word "keyword" containing "key") are treated, and document the choice.</td><td>Writing a precise rule teaches the trade-off between catching attacks and blocking legitimate calls.</td><td>Guard code plus one blocked and one allowed example.</td></tr>
<tr><td align="left">Task 3: Add a new safe tool</td><td>Add a deterministic low-risk tool such as <code>lowercase(text)</code> to <code>TOOLS</code>. Normal: it completes through the runner. Failure: missing arguments should surface as a structured execution error, not a crash.</td><td>Shows that the gates generalise: new tools inherit the guard layer with zero extra safety code.</td><td>Tool code, registry update, and a completed run.</td></tr>
<tr><td align="left">Task 4: Add tests</td><td>Add assert-based tests on a fresh engine covering: a valid call to your tool (normal), a call blocked by your new rule (edge), the stop summary, and an audit-log length or content check (failure detection).</td><td>Tests are what keep the guard layer honest after every future change.</td><td>Passing test cell.</td></tr>
<tr><td align="left">Task 5: Analyse the audit log</td><td>Run a short mixed sequence, display the audit log, and explain in a paragraph which actions were allowed, which were blocked and how you can tell the story from the log alone.</td><td>An audit trail you cannot interpret is decoration; reading it is the operational skill.</td><td>Displayed log plus a short paragraph.</td></tr>
</tbody>
</table>

</div>

<a id="m08e-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook with:

```text
1. Baseline test output.
2. New PreToolUse rule with a blocked and an allowed example.
3. New safe tool and registry update.
4. Added assert-based tests on a fresh engine.
5. Audit-log analysis paragraph.
6. 150–250 word reflection.
```

**Quality checks.** Restart the kernel and run all cells top to bottom before submitting: every assert must pass, your new rule must demonstrably block its target pattern while allowing a normal call, no cell may contain a real secret or credential-like string beyond the teaching denylist terms, and all section headings and anchors must be intact.

**Debugging guide.** If your new rule never triggers, print `json.dumps(args).lower()` inside the guard and check that your pattern actually appears in the flattened text. If it triggers too often, your pattern is a substring of common words — tighten it or match on token boundaries. If the baseline tests fail after your changes, your guard probably blocks one of the original test inputs; run the failing call alone and read the blocked reason. If `blocked_calls` is lower than expected, remember that unknown tools also increment it — count both kinds. And if the audit log seems short, check whether you rebuilt the engine after registering guards, which discards earlier records.

**Reflection questions.**

1. Why must the PreToolUse gate run before the tool-existence check and the tool itself?
2. Why is a blocked call reported as <code>ok: True</code> while an unknown tool is <code>ok: False</code>, and what would go wrong if these were merged?
3. What are the limits of denylist-based guards, and what would an allowlist version of this engine look like?
4. Why should observe-only hooks (PostToolUse, Stop) still write audit records even though they never block?
5. How would you adapt this design to hooks in a real coding agent, where guards are external commands rather than Python functions?

#### Further Readings

- Claude Code hooks reference: <https://code.claude.com/docs/en/hooks>
- Claude Code hooks guide: <https://code.claude.com/docs/en/hooks-guide>
- MCP official introduction: <https://modelcontextprotocol.io/docs/getting-started/intro>
- LangGraph documentation: <https://langchain-ai.github.io/langgraph/>